# payment_risk_v1 exploration

What this notebook does, top to bottom:

1. build (or load) a versioned `payment_risk_v1` training set through Feast and split it by time;
2. look at the data: fraud rate and volume per day, nulls, which features separate fraud;
3. test which features and views earn their place, walk-forward;
4. compare five XGBoost models, from a depth-1 additive baseline to the pipeline's model;
5. fit the pipeline's model and read its feature gain;
6. pick operating thresholds, check the held-out test, calibration and feature importance;
7. run the pipeline's own gate on the result and save the model.

Every data and training step is a function in `risk_model.steps`, shared with the `payment-risk-training` pipeline. All training and scoring runs on the GPU: the notebook cannot hold a GPU claim, so it submits a TrainJob, reads the results and plots them. Negatives are down-sampled, so every metric below uses the stored sample weights and reflects the real fraud rate.

In [ ]:
import json
import os
import tempfile
import uuid
from datetime import date, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from fraud_feast.lake import read_json
from IPython.display import Markdown
from sklearn.metrics import precision_recall_curve

from risk_model import objects, steps
from risk_model.data import DatasetReader, Split
from risk_model.evaluate import DECLINE_FPR, STEP_UP_FPR
from risk_model.split import Window, WindowSplitter
from risk_model.train import BASELINE_PARAMS, load_booster
from risk_model.windows import rolling

## 0. Windows and settings

`rolling` lays out three contiguous windows that end on `END_DAY`, the same rule the pipeline uses. On dev 2026-09-02 to 2026-10-27 is replayed: use `END_DAY = date(2026, 10, 27)` with 36, 9 and 9 days. Override the settings with `NOTEBOOK_END_DAY`, `NOTEBOOK_TRAIN_DAYS`, `NOTEBOOK_VALIDATION_DAYS` and `NOTEBOOK_TEST_DAYS`, the walk-forward fold count with `NOTEBOOK_FOLDS`, and the negative sampling rates with `NOTEBOOK_COMPARE_RATE` and `NOTEBOOK_ABLATION_RATE`. For the small dev dataset, use 0.1 for both rates.

In [ ]:
END_DAY = date.fromisoformat(os.environ.get("NOTEBOOK_END_DAY", "2026-10-28"))
plan = rolling(
    END_DAY,
    train_days=int(os.environ.get("NOTEBOOK_TRAIN_DAYS", "56")),
    validation_days=int(os.environ.get("NOTEBOOK_VALIDATION_DAYS", "14")),
    test_days=int(os.environ.get("NOTEBOOK_TEST_DAYS", "14")),
)
windows = {
    "train": str(plan.train),
    "validation": str(plan.validation),
    "test": str(plan.test),
}
cutoff = plan.cutoff
RANDOM = 7
COMPARE_RATE = float(os.environ.get("NOTEBOOK_COMPARE_RATE", "0.05"))
ABLATION_RATE = float(os.environ.get("NOTEBOOK_ABLATION_RATE", "0.02"))
run_id = str(uuid.uuid4())
splitter = WindowSplitter(**windows)
windows, cutoff

## 1. Training set

Leave `NOTEBOOK_SNAPSHOT` unset to build a new version through Feast, or set it to a recorded snapshot ID to load it.

In [ ]:
snapshot = os.environ.get("NOTEBOOK_SNAPSHOT", "")
if not snapshot:
    planned = steps.plan_examples(**windows, run_id=run_id)
    for chunk in json.loads(planned["chunks"]):
        steps.stage_examples(chunk=chunk, cutoff=cutoff)
    snapshot = steps.assemble_examples(**windows, cutoff=cutoff, run_id=run_id)["snapshot"]
reader = DatasetReader.at(snapshot)
print(json.dumps(reader.provenance(), indent=2))

In [ ]:
print(steps.check_examples(snapshot=snapshot, run_id=run_id))
print(steps.split_windows(snapshot=snapshot, **windows, run_id=run_id))

## 2. The data

`COMPARE_RATE` keeps only a share of the stored negatives in the training split so the first four models in section 4 train quickly. Validation and test keep the stored sample.

In [ ]:
splits = {
    "train": reader.read(splitter["train"], COMPARE_RATE),
    "validation": reader.read(splitter["validation"]),
    "test": reader.read(splitter["test"]),
}
train, validation, test = splits["train"], splits["validation"], splits["test"]
pd.DataFrame(
    {
        name: {
            "rows": len(split.y),
            "positives": int(split.y.sum()),
            "fraud rate (weighted)": float(np.average(split.y, weights=split.w)),
        }
        for name, split in splits.items()
    }
).T

In [ ]:
frame = reader.scan(columns=["started_at", "label"]).to_pandas()
frame["weight"] = np.where(frame["label"].astype(bool), 1.0, 1.0 / reader.sample_rate)
frame["fraud_weight"] = frame["weight"] * frame["label"].astype(float)
frame["day"] = pd.to_datetime(frame["started_at"]).dt.floor("D")
by_day = frame.groupby("day")[["fraud_weight", "weight"]].sum()
by_day["fraud rate"] = by_day["fraud_weight"] / by_day["weight"]

nulls = pd.Series(np.isnan(train.x).mean(axis=0), index=reader.features)
fig, axes = plt.subplots(
    1, 3, figsize=(18, 4.5), gridspec_kw={"width_ratios": [1, 1, 1.5]}, layout="constrained"
)
by_day["fraud rate"].plot(ax=axes[0], marker="o")
axes[0].set_title("Fraud rate per day (weighted)")
by_day["weight"].plot(ax=axes[1], marker="o")
axes[1].set_title("Attempts per day (weighted)")
nulls.sort_values().tail(12).plot(kind="barh", ax=axes[2])
axes[2].set_title("Share of nulls, top 12 features")
plt.show()

In [ ]:
x = pd.DataFrame(train.x, columns=reader.features)
fraud = train.y > 0
gap = (x[fraud].mean() - x[~fraud].mean()) / x.std(ddof=0).replace(0.0, np.nan)
gap.abs().dropna().sort_values().tail(12).plot(kind="barh", figsize=(8, 4.5))
plt.title("Standardized mean gap, fraud vs legit (train sample)")
plt.show()

## 3. Which features earn their place

Fraud arrives in bursts, so one validation day can mislead. Each candidate feature set is scored walk-forward instead: for each of the last `FOLDS` days, a model trains on the days before the previous one, stops early on the previous day and scores the day itself, averaged over three seeds. All fits run in one GPU job. Rows and positives per day come first, because a day with a handful of positives says little. The profile lists features that are constant or whose nulls differ between fraud and legitimate attempts.

A view or derived feature is a candidate to drop only when removing it is no worse on any day and on average. Changing the feature service requires coordinating its training and serving consumers; this notebook only reports the comparison.

In [ ]:
FOLDS = int(os.environ.get("NOTEBOOK_FOLDS", "4"))
SEEDS = (1, 2, 3)
derived = [
    "request_features__customer_amount_to_30d_avg",
    "request_features__payment_method_amount_to_30d_avg",
    "request_features__scored_hour_utc",
]
days = [
    splitter.start + timedelta(days=offset)
    for offset in range((splitter.end - splitter.start).days)
]
daily = {day: reader.read(Window(day, day), ABLATION_RATE) for day in days}


def run(fits):
    """Runs the fits in one GPU job and returns each fit's result by name."""
    spec = json.dumps({"snapshot": snapshot, "fits": fits})
    results = read_json(steps.experiments(run_id=run_id, spec=spec)["results"])
    return {result["name"]: result for result in results["results"]}


def arrays(result):
    """Returns the scores, labels and weights a fit saved for each window it was evaluated on."""
    with tempfile.TemporaryDirectory() as scratch:
        return dict(np.load(objects.download(result["scores"], Path(scratch) / "scores.npz")))


pd.DataFrame(
    {
        str(day): {"rows": len(split.y), "positives": int(split.y.sum())}
        for day, split in daily.items()
    }
).T

In [ ]:
everything = Split.join(list(daily.values()))
table = pd.DataFrame(everything.x, columns=everything.features)
fraud = everything.y > 0


def null_share(rows):
    """Returns each feature's weighted share of nulls among the rows."""
    return table[rows].isna().mul(everything.w[rows], axis=0).sum() / everything.w[rows].sum()


profile = pd.DataFrame(
    {
        "distinct": table.nunique(),
        "null | fraud": null_share(fraud),
        "null | legit": null_share(~fraud),
    }
)
unusual = (profile["distinct"] <= 1) | (profile["null | fraud"] - profile["null | legit"]).abs().gt(
    0.2
)
profile[unusual].round(3)

In [ ]:
assert len(daily) >= FOLDS + 2, (
    "walk-forward needs at least two training days before the first fold"
)
ordered = [str(day) for day in daily]
folds_at = range(len(ordered) - FOLDS, len(ordered))
views = sorted({name.split("__")[0] for name in reader.features})
variants = {"all features": reader.features}
for candidate in derived:
    variants[f"without {candidate.split('__')[1]}"] = [n for n in reader.features if n != candidate]
for view in views:
    variants[f"without view {view}"] = [n for n in reader.features if not n.startswith(f"{view}__")]
results = run(
    [
        {
            "name": f"{label}|{index}|{seed}",
            "features": None if names == reader.features else names,
            "params": {"max_depth": 4},
            "seed": seed,
            "rate": ABLATION_RATE,
            "train": [f"{day}..{day}" for day in ordered[: index - 1]],
            "tune": f"{ordered[index - 1]}..{ordered[index - 1]}",
            "evaluate": {"test": f"{ordered[index]}..{ordered[index]}"},
        }
        for label, names in variants.items()
        for index in folds_at
        for seed in SEEDS
    ]
)
folds = {
    label: np.array(
        [
            np.mean(
                [results[f"{label}|{index}|{seed}"]["metrics"]["test"]["pr_auc"] for seed in SEEDS]
            )
            for index in folds_at
        ]
    )
    for label in variants
}
reference = folds["all features"]
pd.DataFrame(
    {
        label: {
            "mean PR-AUC": values.mean(),
            "change vs all": (values - reference).mean(),
            "days worse": int((values < reference - 1e-9).sum()),
            "worst day change": (values - reference).min(),
        }
        for label, values in folds.items()
    }
).T.round(4).astype({"days worse": int})

## 4. Five models, simple to strong

All five train and score on the GPU in one job. The depth-1 additive model is the baseline the pipeline's gate uses, and the last model is the pipeline's own settings on every stored negative, while the first four use `COMPARE_RATE`. Each model gets its own thresholds from validation: one at 0.1% false-positive rate (decline) and one at 1% (step-up), so the table compares them at the same operating cost.

In [ ]:
candidates = {
    "1. Additive baseline (depth 1)": {"params": BASELINE_PARAMS},
    "2. Random forest": {
        "params": {
            "max_depth": 8,
            "eta": 1.0,
            "subsample": 0.8,
            "colsample_bytree": 1.0,
            "colsample_bynode": 0.6,
            "num_parallel_tree": 200,
            "min_child_weight": 20,
        },
        "rounds": 1,
    },
    "3. Shallow boosting (depth 3)": {"params": {"max_depth": 3}},
    "4. Boosting (depth 5)": {"params": {"max_depth": 5}},
    "5. Pipeline model (depth 6, every stored negative)": {"params": {}, "rate": None},
}
pipeline_model = list(candidates)[-1]
baseline_model = list(candidates)[0]
results = run(
    [
        {
            "name": name,
            "train": [windows["train"]],
            "tune": windows["validation"],
            "evaluate": {"validation": windows["validation"], "test": windows["test"]},
            "seed": RANDOM,
            "rate": COMPARE_RATE,
            **spec,
        }
        for name, spec in candidates.items()
    ]
)
scores = {name: arrays(result) for name, result in results.items()}
thresholds = {name: result["thresholds"] for name, result in results.items()}


def metrics_table(window):
    """Returns a table of metrics per model on a window, best PR-AUC first."""
    table = pd.DataFrame({name: result["metrics"][window] for name, result in results.items()}).T
    return table.sort_values("pr_auc", ascending=False)


def plot_comparison(table, window, title):
    """Plots the metric bars and the precision-recall curves of every model."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
    table[["pr_auc", "recall_at_1pct_fpr"]].plot(kind="bar", ax=axes[0], ylim=(0, 1.05))
    axes[0].set_title(f"{title}: PR-AUC and recall at 1% FPR")
    plt.setp(axes[0].get_xticklabels(), rotation=20, ha="right")
    for name in table.index:
        curve = results[name]["curves"][window]
        axes[1].plot(curve["recall"], curve["precision"], label=name)
    axes[1].set(xlabel="recall", ylabel="precision", title=f"{title}: precision-recall")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


validation_table = metrics_table("validation")
plot_comparison(validation_table, "validation", "Validation")
validation_table.round(4)

## 5. The pipeline's fit

The same TrainJob as the pipeline's `fit_on_gpu` step, which also saves the model file and the additive baseline the gate needs. The model's feature gain is read from the saved file; no scoring happens here.

In [ ]:
fit = steps.fit_on_gpu(snapshot=snapshot, **windows, params="{}", run_id=run_id)
with tempfile.TemporaryDirectory() as scratch:
    booster, order = load_booster(
        objects.download(fit["model_uri"].rstrip("/") + "/model.json", Path(scratch) / "model.json")
    )
assert order == reader.features
read_json(fit["fit"])

## 6. Operating thresholds

The decline threshold sits where 0.1% of legitimate attempts would be declined, and step-up where 1% would. These are the score levels the fraud service can act on.

In [ ]:
validation_scores = scores[pipeline_model]
precision, recall, cuts = precision_recall_curve(
    validation_scores["validation.y"],
    validation_scores["validation"],
    sample_weight=validation_scores["validation.w"],
)
plt.figure(figsize=(8, 4.5))
plt.plot(cuts, precision[:-1], label="precision")
plt.plot(cuts, recall[:-1], label="recall")
for label, value in thresholds[pipeline_model].items():
    plt.axvline(value, ls="--", c="k")
    plt.annotate(f"{label} {value:.3f}", (value, 0.5), rotation=90, va="center", ha="right")
plt.xlabel("score")
plt.legend()
plt.title(f"Validation operating points ({DECLINE_FPR:.1%} and {STEP_UP_FPR:.0%} FPR)")
plt.show()
thresholds[pipeline_model]

## 7. Held-out test

Thresholds come from validation and are applied unchanged to the test window. Lift is the share of fraud caught when only the 5% highest-scoring attempts are reviewed.

In [ ]:
def lift(values, labels, weights, share=0.05):
    """Returns the share of fraud caught by reviewing the top `share` of scores."""
    order = np.argsort(-values)
    weights, labels = weights[order], labels[order] > 0
    reviewed = np.searchsorted(np.cumsum(weights), share * weights.sum())
    return float(weights[:reviewed][labels[:reviewed]].sum() / weights[labels].sum())


test_table = metrics_table("test")
test_table["recall_in_top_5pct"] = pd.Series(
    {name: lift(a["test"], a["test.y"], a["test.w"]) for name, a in scores.items()}
)
plot_comparison(test_table, "test", "Test")
test_table.round(4)

In [ ]:
test_scores = scores[pipeline_model]
flag = test_scores["test"] >= thresholds[pipeline_model]["decline"]
positive = test_scores["test.y"] > 0
weight = test_scores["test.w"]
counts = np.array(
    [
        [weight[~flag & ~positive].sum(), weight[flag & ~positive].sum()],
        [weight[~flag & positive].sum(), weight[flag & positive].sum()],
    ]
)
plt.figure(figsize=(5, 4))
plt.imshow(counts, cmap="Blues")
for (row, column), value in np.ndenumerate(counts):
    plt.text(column, row, f"{value:,.0f}", ha="center", va="center")
plt.xticks([0, 1], ["approve", "decline"])
plt.yticks([0, 1], ["legit", "fraud"])
plt.xlabel("model decision")
plt.ylabel("actual")
plt.title("Test, weighted counts at the decline threshold")
plt.show()

## 8. What the model uses, and how well its scores are calibrated

Gain says which features the trees split on. Calibration compares the mean score in each decile of test scores with the weighted fraud rate in that decile; points near the diagonal mean a score can be read as a probability, which matters when a fixed threshold is enforced.

In [ ]:
gain = pd.Series(booster.get_score(importance_type="gain"))
gain.index = [reader.features[int(key[1:])] for key in gain.index]

frame = pd.DataFrame(
    {
        "score": test_scores["test"],
        "weight": test_scores["test.w"],
        "fraud": test_scores["test.w"] * test_scores["test.y"],
    }
)
frame["decile"] = pd.qcut(frame["score"], 10, duplicates="drop")
by_decile = frame.groupby("decile", observed=True).agg(
    predicted=("score", "mean"), fraud=("fraud", "sum"), weight=("weight", "sum")
)
by_decile["observed"] = by_decile["fraud"] / by_decile["weight"]

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
gain.sort_values().tail(15).plot(kind="barh", ax=axes[0])
axes[0].set_title("Top 15 features by gain (pipeline model)")
axes[1].plot(by_decile["predicted"], by_decile["observed"], marker="o")
axes[1].plot([0, by_decile["predicted"].max()], [0, by_decile["predicted"].max()], ls="--", c="k")
axes[1].set(xlabel="mean score", ylabel="weighted fraud rate", title="Calibration by score decile")
plt.tight_layout()
plt.show()

## 9. The pipeline's own gate, and saving

`steps.gate` is the step the pipeline runs, on the GPU: the model against the depth-1 additive baseline and, once one exists, the production version, on the test window.

In [ ]:
verdict = steps.gate(snapshot=snapshot, **windows, model_uri=fit["model_uri"], run_id=run_id)
summary = json.loads(verdict["summary"])
print("gate passed:", verdict["passed"], summary["reasons"])
pd.DataFrame(summary["metrics"]).T.round(4)

In [ ]:
model_dir = Path(os.path.expanduser("~/models/payment_risk_v1"))
model_dir.mkdir(parents=True, exist_ok=True)
booster.save_model(model_dir / "model.json")

## 10. Summary

In [ ]:
best = validation_table["pr_auc"].idxmax()
model = test_table.loc[pipeline_model]
baseline = test_table.loc[baseline_model]
Markdown(
    f"""
- **Best on validation:** {best}.
- **Pipeline model on test:** PR-AUC {model["pr_auc"]:.3f} against {baseline["pr_auc"]:.3f} for the additive baseline, recall at 1% FPR {model["recall_at_1pct_fpr"]:.3f} against {baseline["recall_at_1pct_fpr"]:.3f}.
- **Review lift:** reviewing the top 5% of scores catches {model["recall_in_top_5pct"]:.0%} of fraud, {model["recall_in_top_5pct"] / 0.05:.1f}x a random review.
- **Thresholds:** decline at score {thresholds[pipeline_model]["decline"]:.3f} (0.1% FPR on validation), step-up at {thresholds[pipeline_model]["step_up"]:.3f} (1% FPR). Start the fraud service's model threshold from the decline value, and revisit it on a larger sample.
- **Gate:** {"passed" if verdict["passed"] else "did not pass"} on this window.
"""
)